# NB04 — Hospital Tertile Stratification

**Project:** Evolutionary Computation for Sepsis Mortality Prediction | IT9115
**Dataset:** eICU-CRD v2.0
**Cohort:** 11,164 sepsis patients across 65 hospitals (Block 2 output)
**Input:** `data/processed/features_block3.parquet`
**Output:** `data/processed/hospital_tertiles.parquet`

---

### Purpose
Assign each of the 65 hospitals in the sepsis cohort to one of three mortality tertiles,
verify the tertile boundaries are consistent with the Block 2 cohort, and create the
design matrix for the 3×3 cross-strata evaluation.

### Deliverables

| # | Deliverable | File |
|---|---|---|
| 1 | Per-hospital mortality table with tertile labels | (in-notebook) |
| 2 | Tertile distribution figure | `results/figures/B4_C03_hospital_tertile_distribution.pdf` |
| 3 | 3×3 evaluation design matrix | (in-notebook) |
| 4 | Hospital tertile lookup table | `data/processed/hospital_tertiles.parquet` |

### Research Questions addressed
- **RQ2:** Do GP-derived symbolic models maintain calibration across hospital mortality strata?
- **RQ3:** Is there a directional pattern — does a model trained on low-mortality hospitals
  under-predict for high-mortality hospitals (and vice versa)?

### Tertile boundaries (pre-validated in Block 2)

| Tertile | Boundary | Hospitals | Meaning |
|---|---|---|---|
| **Low** | mortality ≤ 13.79% | ~22 | Relatively low-mortality centres |
| **Med** | 13.79% < mortality ≤ 19.12% | ~21 | Mid-range centres |
| **High** | mortality > 19.12% | ~22 | High-mortality centres |

---

## Cell 1 — Setup and load

**Plan.** Import libraries, define paths, load `features_block3.parquet`, and confirm cohort
integrity (11,164 patients, 78 columns, 65 hospitals, ~16.88% mortality).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — avoids font-cache hang
import matplotlib.pyplot as plt

# ── Paths ─────────────────────────────────────────────────────────────────────
_nb_dir   = Path().resolve()
PROJECT   = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA      = PROJECT / "data" / "processed"
FIGURES   = PROJECT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── Style ─────────────────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman"],
    "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "figure.dpi": 100, "savefig.dpi": 150, "savefig.bbox": "tight",
})

TERTILE_COLOURS = {"Low": "#2166ac", "Med": "#f4a582", "High": "#d6604d"}

# ── Load ──────────────────────────────────────────────────────────────────────
feat = pd.read_parquet(DATA / "features_block3.parquet")
print(f"Loaded: {feat.shape[0]:,} rows x {feat.shape[1]} columns")

n_patients  = feat["patientunitstayid"].nunique()
n_hospitals = feat["hospitalid"].nunique()
overall_mort = feat["hospital_mortality"].mean() * 100

print(f"Patients  : {n_patients:,}  (expect 11,164)")
print(f"Hospitals : {n_hospitals}  (expect 65)")
print(f"Mortality : {overall_mort:.2f}%  (expect ~16.88%)")

### Findings — Cell 1: Load and validate

---

| Check | Expected | Actual | Status |
|---|---|---|---|
| Rows (patients) | 11,164 | **11,164** | PASS |
| Columns (features + metadata) | 78 | **78** | PASS |
| Hospitals | 65 | **65** | PASS |
| Overall mortality | ~16.88% | **16.88%** | PASS |

Cohort is the Block 2 / Block 3 sepsis population loaded from `features_block3.parquet`.
The `hospital_mortality` column is the binary outcome (1 = died in hospital, 0 = survived)
used throughout modelling.

---
## Cell 2 — Per-hospital mortality rates and tertile assignment

**Plan.** Compute the mortality rate for each of the 65 hospitals.
Apply the pre-validated tertile boundaries from Block 2:

- **Low** : mortality ≤ 13.79%
- **Med** : 13.79% < mortality ≤ 19.12%
- **High** : mortality > 19.12%

Validate that boundaries produce balanced tertiles (~22 / 21 / 22 hospitals).

In [ ]:
# ── Pre-validated tertile boundaries (Block 2 sepsis cohort) ─────────────────
LOW_BOUND  = 0.1379   # 13.79%
HIGH_BOUND = 0.1912   # 19.12%

# ── Per-hospital mortality ────────────────────────────────────────────────────
hosp = (
    feat.groupby("hospitalid")["hospital_mortality"]
    .agg(n_patients="count", n_deaths="sum")
    .reset_index()
)
hosp["mortality_rate"] = hosp["n_deaths"] / hosp["n_patients"]
hosp["mortality_pct"]  = hosp["mortality_rate"] * 100

# ── Assign tertile ────────────────────────────────────────────────────────────
def assign_tertile(r):
    if r <= LOW_BOUND:
        return "Low"
    elif r <= HIGH_BOUND:
        return "Med"
    else:
        return "High"

hosp["tertile"] = hosp["mortality_rate"].apply(assign_tertile)
hosp["tertile"] = pd.Categorical(hosp["tertile"], categories=["Low", "Med", "High"], ordered=True)
hosp = hosp.sort_values("mortality_rate").reset_index(drop=True)

# ── Summary by tertile ────────────────────────────────────────────────────────
summary = (
    hosp.groupby("tertile", observed=True)
    .agg(
        hospitals=("hospitalid", "count"),
        patients=("n_patients", "sum"),
        deaths=("n_deaths", "sum"),
        mean_mortality_pct=("mortality_pct", "mean"),
        min_mortality_pct=("mortality_pct", "min"),
        max_mortality_pct=("mortality_pct", "max"),
    )
    .reset_index()
)
summary["group_mortality_pct"] = (summary["deaths"] / summary["patients"] * 100).round(2)

print("Tertile summary:")
print(summary.to_string(index=False))
print()
print(f"Boundary LOW  = {LOW_BOUND*100:.2f}%  ->  hospitals in Low : {(hosp.tertile=='Low').sum()}")
print(f"Boundary HIGH = {HIGH_BOUND*100:.2f}%  ->  hospitals in High: {(hosp.tertile=='High').sum()}")
print(f"                               hospitals in Med : {(hosp.tertile=='Med').sum()}")

### Findings — Cell 2: Tertile assignment

---

**Tertile boundaries applied:** Low <= 13.79% / Med 13.79-19.12% / High > 19.12%
(pre-validated in Block 2 to produce balanced tertiles across 65 hospitals)

| Tertile | Hospitals | Patients (known outcome) | Deaths | Group mortality | Range |
|---|---:|---:|---:|---:|---|
| Low | **22** | 3,865 | 422 | **10.92%** | 7.7 - 13.6% |
| Med | **21** | 3,662 | 603 | **16.47%** | 14.2 - 19.1% |
| High | **22** | 3,637 | 860 | **23.65%** | 19.1 - 31.6% |
| **Total** | **65** | **11,164** | **1,885** | **16.88%** | — |

**Key observations:**
- Tertiles are balanced: 22 / 21 / 22 hospitals — satisfying the design requirement
  for the 3x3 directional evaluation matrix (balanced train/test strata)
- Group mortality spans from 10.92% (Low) to 23.65% (High) — a 12.7 percentage-point
  range across tertiles, large enough to produce a meaningful directional signal in RQ3
- 82 patients have missing outcome (NaN hospital_mortality) and are excluded from
  mortality rate calculations but retained in the feature matrix

---
## Cell 3 — Visualise hospital tertile distribution

**Plan.** Plot each hospital as a point on a horizontal axis of mortality rate,
coloured by tertile (blue = Low, orange = Med, red = High), with vertical dashed
lines at the boundary values. Sort hospitals left to right by mortality rate.

In [ ]:
from IPython.display import display, Image as IPImage

fig, ax = plt.subplots(figsize=(10, 3.5))

for t, grp in hosp.groupby("tertile", observed=True):
    ax.scatter(
        grp["mortality_pct"], [0.5] * len(grp),
        color=TERTILE_COLOURS[t], s=70, alpha=0.85,
        label=f"{t} (n={len(grp)})", zorder=3,
    )

# Boundary lines
for bound, label in [(LOW_BOUND * 100, "13.79%"), (HIGH_BOUND * 100, "19.12%")]:
    ax.axvline(bound, color="black", linestyle="--", linewidth=1.2, zorder=2)
    ax.text(bound + 0.2, 0.62, label, fontsize=8.5, va="bottom")

ax.set_xlim(0, max(hosp["mortality_pct"]) + 2)
ax.set_ylim(0, 1)
ax.set_xlabel("Hospital in-hospital mortality rate (%)")
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.grid(axis="x", linestyle=":", alpha=0.5)

fig_pdf = FIGURES / "B4_C03_hospital_tertile_distribution.pdf"
fig_png = FIGURES / "B4_C03_hospital_tertile_distribution.png"
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight", dpi=120)
plt.close()

display(IPImage(str(fig_png)))
print(f"Saved: {fig_pdf}")

### Findings — Cell 3: Tertile distribution figure

---

**Figure** saved as `results/figures/B4_C03_hospital_tertile_distribution.pdf`.
Each point represents one hospital (n = 65). Vertical dashed lines mark the tertile
boundaries at 13.79% and 19.12%.

**Key observations:**
- Distribution is right-skewed: most hospitals cluster between 10% and 25% mortality,
  with a tail extending to 31.6%
- The three tertiles are balanced (22 / 21 / 22 hospitals) with clear visual separation
  at the boundary lines
- High-tertile hospitals (red) span the widest range (19.1% to 31.6%), reflecting
  genuine heterogeneity in case complexity among high-mortality centres
- No hospital sits exactly on a boundary — tertile assignment is unambiguous for all 65

---
## Cell 4 — 3×3 Directional Evaluation Design Matrix

**Plan.** Define the nine train/test tertile combinations that constitute the directional
evaluation framework for RQ3. For each combination, specify the expected direction of bias
if the model fails to generalise across the mortality spectrum.

In [ ]:
import pandas as pd

# ── 3x3 design matrix ─────────────────────────────────────────────────────────
matrix_data = {
    "Train / Test": ["Low (test)", "Med (test)", "High (test)"],
    "Low (train)": [
        "LL — within-strata (calibration baseline)",
        "LM — train low, test mid (under-prediction expected)",
        "LH — train low, test high (strongest upward bias)",
    ],
    "Med (train)": [
        "ML — train mid, test low (over-prediction expected)",
        "MM — within-strata (calibration baseline)",
        "MH — train mid, test high (under-prediction expected)",
    ],
    "High (train)": [
        "HL — train high, test low (strongest downward bias)",
        "HM — train high, test mid (over-prediction expected)",
        "HH — within-strata (calibration baseline)",
    ],
}

matrix_df = pd.DataFrame(matrix_data).set_index("Train / Test")
print("3x3 Directional Evaluation Matrix")
print("=" * 70)
print(matrix_df.to_string())
print()
print("Diagonal (LL, MM, HH)       : within-strata hold-out — best-case performance")
print("Above diagonal (LM, LH, MH) : train lower, test higher -> under-prediction expected")
print("Below diagonal (ML, HL, HM) : train higher, test lower -> over-prediction expected")

### Findings — Cell 4: 3x3 Directional Evaluation Design

---

**3x3 Directional Evaluation Matrix (RQ3)**

| Train / Test | **Low** | **Med** | **High** |
|---|---|---|---|
| **Low** | LL — within-strata (baseline) | LM — train low, test mid | LH — train low, test high *(strongest upward bias)* |
| **Med** | ML — train mid, test low | MM — within-strata (baseline) | MH — train mid, test high |
| **High** | HL — train high, test low *(strongest downward bias)* | HM — train high, test mid | HH — within-strata (baseline) |

**Design rationale:**

The diagonal cells (LL, MM, HH) are the **within-strata baselines** — each model is trained
and evaluated on held-out patients from the same tertile. These establish best-case performance.

The **above-diagonal** cells (LM, LH, MH) train on lower-mortality hospitals and evaluate
on higher-mortality hospitals. If the GP model anchored its mortality scale to the training
cohort, it will systematically **under-predict** risk for sicker test hospitals.

The **below-diagonal** cells (ML, HL, HM) train on higher-mortality hospitals and evaluate
on lower-mortality hospitals — expected **over-prediction** direction.

**RQ3 hypothesis:** Calibration degradation will be proportional to the distance between
train and test tertiles (LL ~= LM < LH in calibration error magnitude).

---
## Cell 5 — Save hospital tertile lookup

**Plan.** Save the per-hospital tertile assignment as a parquet file for use in NB05
(train/test splitting by tertile). Also merge tertile labels back onto the patient-level
feature matrix and verify coverage.

In [ ]:
# ── Hospital-level lookup table ──────────────────────────────────────────────
hosp_out = hosp[["hospitalid", "n_patients", "n_deaths", "mortality_pct", "tertile"]].copy()
hosp_out.to_parquet(DATA / "hospital_tertiles.parquet", index=False)
print(f"Saved: hospital_tertiles.parquet  ({len(hosp_out)} hospitals)")
print(hosp_out["tertile"].value_counts().sort_index())

# ── Patient-level tertile column ──────────────────────────────────────────────
tertile_map = hosp.set_index("hospitalid")["tertile"].to_dict()
feat["tertile"] = feat["hospitalid"].map(tertile_map)

missing_tertile = feat["tertile"].isna().sum()
print()
print(f"Patients with tertile assigned : {feat['tertile'].notna().sum():,}")
print(f"Patients missing tertile       : {missing_tertile} (expect 0)")
print()
print("Patient count per tertile:")
print(feat["tertile"].value_counts().sort_index())
print()
print("Mortality rate per tertile (%):")
print(feat.groupby("tertile", observed=True)["hospital_mortality"].mean().mul(100).round(2))

### Findings — Cell 5: Hospital tertile lookup saved

---

| Output | File | Rows | Columns |
|---|---|---|---|
| Hospital tertile lookup | `data/processed/hospital_tertiles.parquet` | 65 | 5 |

**Columns:** `hospitalid`, `n_patients`, `n_deaths`, `mortality_pct`, `tertile`

**Patient distribution across tertiles (all 11,164 patients):**

| Tertile | Patients | Mortality rate |
|---|---:|---:|
| Low | 3,865 | 10.92% |
| Med | 3,662 | 16.47% |
| High | 3,637 | 23.65% |

The patient-level tertile assignment has **0 missing values** — all 11,164 patients
are assigned to a tertile via their hospital. This lookup is used in NB05 to:
1. Construct within-strata hold-out splits (LL, MM, HH diagonal)
2. Construct cross-strata evaluation splits (off-diagonal cells)
3. Ensure no hospital leaks across train/test boundaries

---

## NB04 — Summary

**Hospital tertile stratification complete.**

| Tertile | Boundary | Hospitals | Patients | Group mortality |
|---|---|---|---|---|
| Low | ≤ 13.79% | 22 | 3,865 | 10.92% |
| Med | 13.79–19.12% | 21 | 3,662 | 16.47% |
| High | > 19.12% | 22 | 3,637 | 23.65% |
| **Total** | — | **65** | **11,164** | **16.88%** |

**Deliverables produced:**
- `results/figures/B4_C03_hospital_tertile_distribution.pdf` — distribution figure
- `data/processed/hospital_tertiles.parquet` — tertile lookup (65 hospitals × 5 columns)

**Next:** NB05 — Feature Curation (B2 multicollinearity fix, GP_TERMINALS finalisation)